# 08 · Business recommendations
Questions 16–19, sized from the analysis.

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import get_engine, query, PROJECT_ROOT
from src import analysis as an
an.set_style()
pd.set_option("display.float_format", "{:,.2f}".format); pd.set_option("display.max_columns", 30)
engine = get_engine()
o = an.sql_output

In [2]:
o('q15_declining_customers')

,engagement_status,value_segment,customers,monthly_revenue_before_kes,monthly_revenue_now_kes,annualised_revenue_at_risk_kes
0,Declining (>50% revenue drop),Silver (next 30%),178,"123,793.00","37,141.00","1,039,824.00"
1,Declining (>50% revenue drop),Gold (next 20%),133,"120,292.00","34,235.00","1,032,688.00"
2,Declining (>50% revenue drop),Bronze (bottom 40%),349,"96,211.00","30,442.00","789,228.00"
3,Declining (>50% revenue drop),Platinum (top 10%),40,"72,357.00","20,307.00","624,596.00"
4,Silent (no activity Aug-Oct),Bronze (bottom 40%),2,394.00,0.00,"4,732.00"


In [3]:
o('q23_valuable_customers_at_risk').head(10)

,customer_id,customer_type,plan_type,county,age_group,value_segment,engagement_status,monthly_revenue_may_jul_kes,monthly_revenue_aug_oct_kes,open_complaints,priority
0,C005343,Postpaid,Postpaid Business,Kisumu,35-44,Platinum (top 10%),Declining (>50% revenue drop),"5,299.00","1,371.00",0,1
1,C022979,Postpaid,Postpaid Business,Turkana,45-54,Platinum (top 10%),Declining (>50% revenue drop),"3,689.00",384.00,0,2
2,C020773,Postpaid,Postpaid Business,Kilifi,35-44,Platinum (top 10%),Declining (>50% revenue drop),"2,957.00",314.00,0,3
3,C018658,Postpaid,Postpaid Business,Kiambu,35-44,Platinum (top 10%),Declining (>50% revenue drop),"3,513.00",954.00,0,4
4,C012892,Postpaid,Postpaid Business,Mombasa,35-44,Platinum (top 10%),Declining (>50% revenue drop),"3,455.00","1,005.00",0,5
5,C023713,Postpaid,Postpaid Business,Kakamega,25-34,Platinum (top 10%),Declining (>50% revenue drop),"3,669.00","1,554.00",0,6
6,C017245,Postpaid,Postpaid Business,Nakuru,25-34,Platinum (top 10%),Declining (>50% revenue drop),"2,906.00",906.00,0,7
7,C017048,Postpaid,Postpaid Business,Kilifi,35-44,Platinum (top 10%),Declining (>50% revenue drop),"2,318.00",329.00,0,8
8,C010045,Postpaid,Postpaid Business,Nairobi,45-54,Platinum (top 10%),Declining (>50% revenue drop),"3,043.00","1,133.00",0,9
9,C005490,Postpaid,Postpaid Business,Nairobi,25-34,Platinum (top 10%),Declining (>50% revenue drop),"3,054.00","1,237.00",0,10


In [4]:
pd.concat([o('q21_campaign_response').set_index('campaign_type'), ], axis=1)

,contacts,accepted_pct,opted_out_pct,response_unknown_pct
campaign_type,,,,
Loyalty points,7098,34.40,6.30,2.00
Win-back offer,7601,29.50,10.20,2.10
Data bonus offer,11007,24.00,7.40,2.00
Bundle upsell,9817,11.80,8.90,1.90


In [5]:
o('q22_winback_effect')

,response,contacts,closed_within_6m,closed_within_6m_pct
0,Accepted offer,1865,31,1.70
1,Did not accept,4388,4110,93.70


**Win-back caution.** Customers who *accepted* a win-back offer almost all stayed, while those who didn't mostly left. This comparison is **self-selected**: customers who accept were probably less determined to leave. It doesn't prove the offer caused the retention. A holdout group (declining customers deliberately not contacted) would measure the true effect. It is cheap to set up, and it is the first thing to fix in campaign design.

In [6]:
p = o("q25_price_change_churn")
pv = p.pivot_table(index=["customer_type", "value_segment"], columns="period", values="monthly_churn_pct")
pv["after_vs_before"] = (pv.iloc[:, 1] - pv.iloc[:, 0]).round(2); pv.sort_values("after_vs_before", ascending=False)

period                             a) Before (Mar-Oct 2025)  \
customer_type value_segment                                   
Prepaid       Bronze (bottom 40%)                      2.37   
              Silver (next 30%)                        1.13   
              Gold (next 20%)                          1.17   
Postpaid      Bronze (bottom 40%)                      1.22   
              Platinum (top 10%)                       0.33   
              Silver (next 30%)                        0.54   
Prepaid       Platinum (top 10%)                       0.62   
Postpaid      Gold (next 20%)                          0.71   

period                             b) After (Nov 2025-Feb 2026)  \
customer_type value_segment                                       
Prepaid       Bronze (bottom 40%)                          3.62   
              Silver (next 30%)                            1.42   
              Gold (next 20%)                              1.23   
Postpaid      Bronze (bottom 40%)                          1.23   
              Platinum (top 10%)                           0.32   
              Silver (next 30%)                            0.53   
Prepaid       Platinum (top 10%)                           0.55   
Postpaid      Gold (next 20%)                              0.40   

period                             c) Later (Mar-Oct 2026)  after_vs_before  
customer_type value_segment                                                  
Prepaid       Bronze (bottom 40%)                     2.15             1.25  
              Silver (next 30%)                       1.26             0.29  
              Gold (next 20%)                         1.20             0.06  
Postpaid      Bronze (bottom 40%)                     1.38             0.01  
              Platinum (top 10%)                      0.35            -0.01  
              Silver (next 30%)                       0.70            -0.01  
Prepaid       Platinum (top 10%)                      0.89            -0.07  
Postpaid      Gold (next 20%)                         0.76            -0.31

Closures after the price change rose most among **prepaid Bronze** customers (low early spend). Higher-value and postpaid customers barely moved. The extra closures were concentrated in Nov 2025–Feb 2026, about four months after the change, and churn returned towards its earlier level afterwards.

In [7]:
k = o("q01_headline_kpis").iloc[0]
seg = o("q15_declining_customers")
at_risk = seg.annualised_revenue_at_risk_kes.sum()
hv = seg[seg.value_segment.str.startswith(("Platinum", "Gold"))].annualised_revenue_at_risk_kes.sum()
annual_rev = k.revenue * 12
print(f"Declining customers today: {seg.customers.sum():,}; annualised revenue already lost from their decline: KES {at_risk/1e6:.1f}M")
print(f"  of which Platinum/Gold: KES {hv/1e6:.1f}M ({100*hv/at_risk:.0f}%)")
print(f"Run-rate annual revenue (Oct 2026 x 12): KES {annual_rev/1e6:.0f}M")

Declining customers today: 702; annualised revenue already lost from their decline: KES 3.5M
  of which Platinum/Gold: KES 1.7M (47%)
Run-rate annual revenue (Oct 2026 x 12): KES 215M


## Recommendations

| # | Recommendation | Evidence | Measure |
|---|---|---|---|
| 1 | **Monthly early-warning list**: contact customers flagged *Declining* before the 90-day closure, Platinum/Gold first | Churn is preceded by about 2 months of falling activity; ~700 customers are in decline now, a quarter of them Platinum/Gold, who carry about half the revenue at risk (q15, q23) | Share of flagged customers active again after 60 days |
| 2 | **Fix network complaints first** in Turkana, Kilifi and Garissa, and set a resolution target for network and data-speed tickets | These counties have about 2× the complaint rate and the highest churn; network tickets take about 12 days and 30% are never resolved (q17, q19, q9) | Unresolved network-ticket share; county churn |
| 3 | **Resolve-or-escalate rule** for every complaint over 7 days old | Unresolved complainers churn about 1.2–1.5× more than non-complainers within the same value tier (q18) | % of tickets closed ≤7 days |
| 4 | **Protect low-spend prepaid customers in future price changes** (keep an entry bundle at the old price) | Closures of prepaid Bronze customers rose from about 2.4% to 3.6% a month after the July 2025 rise (q25) | Churn of the low-spend prepaid segment after the next tariff change |
| 5 | **Grow data**: target data-heavy and balanced prepaid users with data and combo upsells; don't push data offers to voice-first users | Data per user up ~25% YoY; voice and SMS flat (q24); behavioural clusters show different needs within each value tier (nb 06) | Data bundle revenue per active customer |
| 6 | **Add a holdout group to win-back campaigns** | The current comparison is self-selected (q22) | Measured incremental retention |

**What the data cannot tell us:** why individual customers leave (no exit survey), competitor offers, network quality measured directly (only complaints), and contribution margin (CLV is shown as a range).